# 🛡️ Model A: Custom Transformer Classifier from Scratch
### Course Project: Transformer — From Scratch vs. Pretrained Model (Member 1)
**Task:** Binary Spam / Toxicity Detection (Ham vs. Spam)
**Hardware Target:** Free Google Colab T4 GPU

---
### 📋 Academic Integrity & Grading Compliance
- **Pure PyTorch Primitives Only:** Built exclusively using `nn.Module`, `nn.Linear`, `nn.Embedding`, `nn.Dropout`, `nn.LayerNorm`, and tensor math.
- **Strictly Prohibited:** `nn.Transformer`, `nn.TransformerEncoder`, `nn.TransformerEncoderLayer`, `nn.MultiheadAttention`, and pretrained embeddings are **NOT used**.
- **Zero Data Leakage:** Tokenizer and vocabulary are fitted **strictly on `train.csv`**.
- **Held-Out Test Set:** `test.csv` is evaluated once and remains completely untouched during training and hyperparameter tuning.

## 1. Hardware Verification & GPU Setup
Verify that Google Colab is running with a GPU enabled (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
import torch
import sys
import os

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
device_available = torch.cuda.is_available()
print(f"CUDA Available: {device_available}")

if device_available:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Active GPU: {gpu_name}")
else:
    gpu_name = "CPU"
    print("Running on CPU (For optimal speed, enable GPU under Runtime -> Change runtime type)")

## 2. Clone Repository & Install Dependencies

In [ ]:
# If running inside Google Colab, clone the repo if not already in workspace
if not os.path.exists("models"):
    !git clone https://github.com/amrrtarrek/transformer-spam-toxicity-comparison.git
    %cd transformer-spam-toxicity-comparison

!pip install -q -r requirements.txt

## 3. Deterministic Data Preparation & Stratified Splitting
Downloads the UCI SMS Spam Collection dataset, fixes `SEED=42`, and creates deterministic stratified splits: 80% Train, 10% Validation, 10% Test.

In [ ]:
!python scripts/data_prep.py

### Inspect Dataset Distribution & Metadata

In [ ]:
import json
import pandas as pd

with open("data/dataset_metadata.json", "r") as f:
    meta = json.load(f)

print(f"Dataset: {meta['dataset_name']}")
print(f"Citation: {meta['citation']}")
print(f"Overall: {meta['overall_distribution']}")
print("\nSplits Distribution:")
for split, dist in meta['splits_distribution'].items():
    print(f"  - {split}: Total={dist['total']} | Counts={dist['counts']} | Percentages={dist['percentages']}")

train_df = pd.read_csv("data/train.csv")
print("\nTrain Head Sample:")
display(train_df.head(3))

## 4. Model Architecture & Tokenizer Verification
Verify that Model A is constructed strictly with custom PyTorch primitives (no forbidden modules).

In [ ]:
from models.scratch.transformer import TransformerClassifier
from models.scratch.tokenizer import SimpleTokenizer

# Fit tokenizer strictly on training set
tok = SimpleTokenizer(max_length=128)
tok.fit_from_csv("data/train.csv", max_vocab_size=8000)
tok.save("artifacts/scratch_tokenizer.json")

# Instantiate Model A
model = TransformerClassifier(
    vocab_size=tok.vocab_size,
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=256,
    max_len=128,
    num_classes=2,
    dropout=0.1,
    pad_idx=tok.pad_id,
    pooling="mean"
)

print("\n=== Model A Architecture Overview ===")
print(model)
print(f"\nTotal Trainable Parameters: {model.count_parameters():,}")

## 5. Train Model A (Optimized for Free Colab T4 GPU)
Trains on `data/train.csv`, tunes on `data/val.csv`, applies early stopping, and checkpoints the best weights to `artifacts/model_a_scratch.pt`.

In [ ]:
from scripts.train_scratch import train_model

train_summary = train_model(
    train_csv="data/train.csv",
    val_csv="data/val.csv",
    artifacts_dir="artifacts",
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=256,
    max_length=128,
    batch_size=64,
    lr=3e-4,
    epochs=15,
    patience=4
)
print("\nTraining Summary:", train_summary)

## 6. Evaluate Model A on Held-Out Test Set (`data/test.csv`)
Computes Accuracy, Macro-F1, Precision, Recall, Latency, and saves `artifacts/model_a_metrics.json` and `artifacts/model_a_confusion_matrix.png`.

In [ ]:
from scripts.evaluate import evaluate_test_set
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

metrics = evaluate_test_set(
    test_csv="data/test.csv",
    model_path="artifacts/model_a_scratch.pt",
    tokenizer_path="artifacts/scratch_tokenizer.json",
    training_time_seconds=train_summary["training_time_seconds"],
    hardware_name=gpu_name
)

print("\n=== Held-Out Test Set Metrics ===")
print(json.dumps(metrics, indent=2))

# Display Confusion Matrix Plot
if os.path.exists("artifacts/model_a_confusion_matrix.png"):
    img = mpimg.imread("artifacts/model_a_confusion_matrix.png")
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

## 7. Interactive Prediction & Member 3 Handoff Verification
Demonstrates the standardized `ScratchModelPredictor` interface for Member 3's Streamlit dashboard.

In [ ]:
from models.scratch.inference import ScratchModelPredictor

predictor = ScratchModelPredictor(
    model_path="artifacts/model_a_scratch.pt",
    tokenizer_path="artifacts/scratch_tokenizer.json"
)

test_samples = [
    "Hey are we still meeting for lunch today at 12?",
    "URGENT! You have won a £1,000 cash prize or a guaranteed 7-day holiday! Call 09061701461 to claim now.",
    "Can you send me the lecture notes for chapter 4?",
    "FREE ringtone! Text YES to 80082 now to get 5 free tones!"
]

for sample in test_samples:
    res = predictor.predict(sample)
    print(f"\nInput: '{sample}'")
    print(f"  -> Pred: {res['label']} (Confidence: {res['confidence']*100:.2f}%, Latency: {res['latency_ms']:.2f} ms)")
    print(f"  -> Probabilities: {res['probabilities']}")

## 8. Export Generated Artifacts
Optionally download or commit the generated artifacts back to your repository.

In [ ]:
try:
    from google.colab import files
    files.download("artifacts/model_a_metrics.json")
    files.download("artifacts/model_a_confusion_matrix.png")
    print("Artifacts downloaded successfully.")
except ImportError:
    print("Not running in Colab; artifacts saved locally under artifacts/")